### Step 1: Load and Explore Dataset
In this step, we load the dataset into Pandas and explore its structure.
We check the first and last rows, dataset shape, column names, and data types
to understand what kind of preprocessing is required.

In [12]:
import pandas as pd

df = pd.read_csv("student_data_100_preprocess.csv")

print("First 5 rows:")
print(df.head())

print("\nLast 5 rows:")
print(df.tail())

print("\nShape:", df.shape)

print("\nColumns:", df.columns.tolist())

print("\nData Types:\n", df.dtypes)

num_cols = df.select_dtypes(include=['int64','float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns
print("\nNumerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

print("\nStatistical Summary:\n", df[num_cols].describe())


First 5 rows:
   Gender   Age  Hours_Studied  Attendance Parental_Education  Final_Score
0    Male   NaN            1.0        65.0           Graduate         95.0
1  Female  21.0            1.0        80.0           Graduate         70.0
2    Male   NaN            3.0        60.0       Postgraduate         60.0
3    Male  21.0            2.0        60.0        High School         80.0
4    Male  20.0            5.0        80.0           Graduate          NaN

Last 5 rows:
    Gender   Age  Hours_Studied  Attendance Parental_Education  Final_Score
95  Female  18.0            6.0         NaN       Postgraduate         65.0
96  Female  20.0            6.0        95.0        High School          NaN
97  Female  20.0            3.0        80.0                NaN         75.0
98  Female  18.0            6.0        60.0                NaN         50.0
99    Male  19.0            8.0        90.0                NaN         90.0

Shape: (100, 6)

Columns: ['Gender', 'Age', 'Hours_Studied', 'Att

### Step 2: Identify Missing Values
In this step, we check for missing values in each column.
This helps us decide how to handle them (mean for numerical columns, mode for categorical columns).


In [13]:
print("Missing values per column:\n", df.isnull().sum())

Missing values per column:
 Gender                 0
Age                    9
Hours_Studied         10
Attendance             8
Parental_Education    13
Final_Score            7
dtype: int64


### Step 3: Handle Missing Values
We fill missing values to ensure the dataset is complete.
Numerical columns are filled with their mean, while categorical columns are filled with their mode.
This prevents errors during model training and keeps the dataset consistent.


In [14]:
df.fillna({
    'Age': df['Age'].mean(),
    'Hours_Studied': df['Hours_Studied'].mean(),
    'Attendance': df['Attendance'].mean(),
    'Final_Score': df['Final_Score'].mean(),
    'Gender': df['Gender'].mode()[0],
    'Parental_Education': df['Parental_Education'].mode()[0]
}, inplace=True)

print("Missing values after filling:\n", df.isnull().sum())

print("\nDataset Info:")
print(df.info())
print("\nStatistical Summary:\n", df.describe())


Missing values after filling:
 Gender                0
Age                   0
Hours_Studied         0
Attendance            0
Parental_Education    0
Final_Score           0
dtype: int64

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Gender              100 non-null    object 
 1   Age                 100 non-null    float64
 2   Hours_Studied       100 non-null    float64
 3   Attendance          100 non-null    float64
 4   Parental_Education  100 non-null    object 
 5   Final_Score         100 non-null    float64
dtypes: float64(4), object(2)
memory usage: 4.8+ KB
None

Statistical Summary:
               Age  Hours_Studied  Attendance  Final_Score
count  100.000000     100.000000  100.000000   100.000000
mean    19.417582       4.366667   76.684783    74.247312
std      1.119758       2.226788   11.354641    

### Step 4: Encode Categorical Variables
We convert categorical columns into numerical format.
Gender is encoded using Label Encoding, while Parental_Education is encoded using One-Hot Encoding.
This ensures the dataset contains only numerical values for machine learning.


In [15]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

df = pd.get_dummies(df, columns=['Parental_Education'], drop_first=True)

print("\nDataset Info after Encoding:")
print(df.info())

print("\nFirst 5 rows after encoding:")
print(df.head())



Dataset Info after Encoding:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Gender                           100 non-null    int64  
 1   Age                              100 non-null    float64
 2   Hours_Studied                    100 non-null    float64
 3   Attendance                       100 non-null    float64
 4   Final_Score                      100 non-null    float64
 5   Parental_Education_High School   100 non-null    bool   
 6   Parental_Education_Postgraduate  100 non-null    bool   
dtypes: bool(2), float64(4), int64(1)
memory usage: 4.2 KB
None

First 5 rows after encoding:
   Gender        Age  Hours_Studied  Attendance  Final_Score  \
0       1  19.417582            1.0        65.0    95.000000   
1       0  21.000000            1.0        80.0    70.000000   
2       1  19.417582   

### Step 5: Prepare Data for Model Training
We separate the dataset into features (X) and target (y).
Then we split the data into training (80%) and testing (20%) sets.
This ensures the model is trained on one part of the data and evaluated on another, improving reliability.


In [16]:
from sklearn.model_selection import train_test_split

X = df.drop('Final_Score', axis=1)
y = df['Final_Score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


Training set shape: (80, 6)
Testing set shape: (20, 6)
